<a href="https://colab.research.google.com/github/springboardmentor1979b-cmyk/AI_Company-Internal-Chatbot-with-Role-Based-Access-Control/blob/Dhanishka_work/Company_Internal_Chatbot_with_RBAC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi streamlit langchain sentence-transformers pandas pyjwt

In [ ]:
!git clone https://github.com/springboardmentor441p-coderr/Fintech-data

In [ ]:
!ls Fintech-data

In [ ]:
import pandas as pd
df = pd.read_csv("Fintech-data/HR/hr_data.csv")
df.head(10)

In [ ]:
import os
import pandas as pd

In [ ]:
def clean_text(text):
    # Remove special characters
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.strip()
    # Normalize multiple spaces
    text = " ".join(text.split())
    return text

In [ ]:
folder = "Fintech-data"
for file in os.listdir(folder):
    if file.endswith(".md"):
        with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
            raw_text = f.read()
            cleaned = clean_text(raw_text)
            print(f"--- {file} ---")
            print(cleaned[:300])  # show first 300 chars

In [ ]:
for file in os.listdir(folder):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        # Apply cleaning to text columns
        for col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].apply(lambda x: clean_text(str(x)))
        print(f"--- {file} ---")
        print(df.head())

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize

def chunk_text(text, chunk_size=300):
    words = word_tokenize(text)
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

In [ ]:
folder = "Fintech-data"
for file in os.listdir(folder):
    if file.endswith(".md"):
        with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
            raw_text = f.read()
            chunks = chunk_text(raw_text, chunk_size=300)
            print(f"--- {file} ---")
            print(f"Total chunks: {len(chunks)}")
            print("Sample chunk:\n", chunks[0][:200])

In [ ]:
for file in os.listdir(folder):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder, file))
        for col in df.columns:
            if df[col].dtype == "object":
                text = " ".join(df[col].dropna().astype(str).tolist())
                chunks = chunk_text(text, chunk_size=300)
                print(f"--- {file} ---")
                print(f"Total chunks: {len(chunks)}")
                print("Sample chunk:\n", chunks[0][:200])

In [ ]:
role_mapping = {
    "finance": ["finance_report.md", "quarterly_summary.csv"],
    "marketing": ["marketing_analysis.md", "campaign_data.csv"],
    "hr": ["employee_data.csv", "hr_policies.md"],
    "engineering": ["tech_docs.md", "system_architecture.md"],
    "employees": ["general_handbook.md"],
    "c_level": "all"  # C-Level can access everything
}


In [ ]:
def assign_metadata(file_name, chunk, role_mapping):
    metadata = {}
    metadata["source_file"] = file_name
    metadata["chunk_text"] = chunk
    metadata["roles"] = []

    for role, files in role_mapping.items():
        if files == "all" or file_name in files:
            metadata["roles"].append(role)
    return metadata

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
!pip install ipywidgets

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


In [ ]:
import chromadb

# Initialize a persistent ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Get or create the collection (this will now point to the persistent storage)
collection = client.get_or_create_collection(name="company_docs")

print(f"ChromaDB collection initialized with {collection.count()} items.")

In [ ]:
all_chunks_with_metadata = []

for root, dirs, files in os.walk(folder):
    for file_name in files:
        file_path = os.path.join(root, file_name)

        if file_name.endswith(".md"):
            with open(file_path, "r", encoding="utf-8") as f:
                raw_text = f.read()
                cleaned_text = clean_text(raw_text)
                chunks = chunk_text(cleaned_text, chunk_size=300)
                for chunk in chunks:
                    # Pass only the file_name, not full path, for metadata consistency
                    all_chunks_with_metadata.append(assign_metadata(file_name, chunk, role_mapping))

        elif file_name.endswith(".csv"):
            df = pd.read_csv(file_path)
            for col in df.columns:
                if df[col].dtype == "object":
                    text_for_chunking = " ".join(df[col].dropna().astype(str).tolist())
                    cleaned_text = clean_text(text_for_chunking)
                    chunks = chunk_text(cleaned_text, chunk_size=300)
                    for chunk in chunks:
                        # Pass only the file_name, not full path, for metadata consistency
                        all_chunks_with_metadata.append(assign_metadata(file_name, chunk, role_mapping))

# Ensure the ChromaDB collection is empty before adding to avoid duplicates on re-run
# If it's a fresh run, collection.count() will be 0, otherwise clear it.
if collection.count() > 0:
    collection.delete(where={})

for i, item in enumerate(all_chunks_with_metadata):
    text = item["chunk_text"]
    roles = item["roles"]
    source = item["source_file"]

    embedding = model.encode(text).tolist()

    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{"roles": roles, "source": source}]
    )
print(f"Added {len(all_chunks_with_metadata)} chunks to ChromaDB.")

In [ ]:
results = collection.query(
    query_texts=["latest quarterly spend"],
    n_results=3
)
print(results)

In [ ]:
def role_based_search(query, user_role, top_k=3):
    # Run semantic search
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    # Filter results by role
    filtered_docs = []
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        if user_role in meta["roles"] or "c_level" in meta["roles"]:
            filtered_docs.append({
                "document": doc,
                "source": meta["source"],
                "roles": meta["roles"]
            })
    return filtered_docs


In [ ]:
if all_chunks_with_metadata:
    print(all_chunks_with_metadata[0])
else:
    print("The list all_chunks_with_metadata is empty.")


In [ ]:
for item in all_chunks_with_metadata[:5]:
    print(item["source_file"], item["roles"])

In [ ]:
for i, item in enumerate(all_chunks_with_metadata):
    text = item["chunk_text"]
    roles = item["roles"]
    source = item["source_file"]

    embedding = model.encode(text).tolist()

    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{"roles": roles, "source": source}]
    )

In [ ]:
def role_based_search(query, user_role, top_k=3):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    filtered_docs = []
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        if user_role in meta["roles"] or "c_level" in meta["roles"]:
            filtered_docs.append({
                "document": doc,
                "source": meta["source"],
                "roles": meta["roles"]
            })
    return filtered_docs

finance_results = role_based_search("latest quarterly spend", "finance")
print(finance_results)

### Streamlit Frontend for Role-Based Search



In [ ]:
!pip install pyngrok

from pyngrok import ngrok

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

# Get the authtoken from Colab Secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set successfully!")
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it.")

In [ ]:
import chromadb

# Initialize a persistent ChromaDB client
client = chromadb.PersistentClient(path="./chroma_db")

# Get or create the collection (this will now point to the persistent storage)
collection = client.get_or_create_collection(name="company_docs")

print(f"ChromaDB collection initialized with {collection.count()} items.")

In [ ]:
%%writefile app.py
import streamlit as st
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

st.set_page_config(layout="wide", page_title="Company Document Search")

# Initialize model and ChromaDB client
# These are re-initialized for the Streamlit app to run standalone
@st.cache_resource
def get_model():
    return SentenceTransformer("all-MiniLM-L6-v2")

@st.cache_resource
def get_chroma_collection():
    client = chromadb.PersistentClient(path="./chroma_db")
    return client.get_or_create_collection(name="company_docs")

model = get_model()
collection = get_chroma_collection()

# Ensure the collection is populated if it's empty
if collection.count() == 0:
    st.warning("ChromaDB collection is empty. Please ensure the data ingestion step was run in the notebook.")

def role_based_search(query, user_role, top_k=5):
    # Embed the query
    query_embedding = model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    filtered_docs = []
    if results and results["documents"]:
        for i, doc in enumerate(results["documents"][0]):
            meta = results["metadatas"][0][i]
            # Check if user_role has access OR if 'c_level' is among the roles that have access
            if user_role in meta["roles"] or "c_level" in meta["roles"]:
                filtered_docs.append({
                    "document": doc,
                    "source": meta["source"],
                    "roles": meta["roles"]
                })
    return filtered_docs

# --- Session State Management ---
if 'page' not in st.session_state:
    st.session_state.page = 'role_selection'
if 'selected_role' not in st.session_state:
    st.session_state.selected_role = None
if 'search_query_input' not in st.session_state:
    st.session_state.search_query_input = ""
if 'current_search_results' not in st.session_state:
    st.session_state.current_search_results = []

# --- Sample Queries for different roles ---
sample_queries_by_role = {
    'finance': ['latest quarterly spend', 'budget allocation for Q3', 'financial performance metrics'],
    'marketing': ['best performing campaigns', 'marketing spend ROI', 'customer acquisition strategies'],
    'hr': ['employee leave policy', 'performance review guidelines', 'new hire onboarding process'],
    'engineering': ['system architecture diagrams', 'coding standards', 'deployment procedures'],
    'employees': ['general company policies', 'holiday schedule', 'benefits information'],
    'c_level': ['overall company performance', 'strategic initiatives 2025', 'market trends analysis']
}

# --- UI Functions ---
def show_role_selection_page():
    st.title("🏢 Company Document Search")
    st.markdown("### Welcome! Please select your role to begin searching internal documents.")

    available_roles = ['finance', 'marketing', 'hr', 'engineering', 'employees', 'c_level']
    user_role_selection = st.selectbox("Select your access role:", available_roles, index=available_roles.index('employees'))

    if st.button("Proceed to Search"):
        st.session_state.selected_role = user_role_selection
        st.session_state.page = 'query_and_results'
        # Removed st.experimental_rerun()

def show_query_and_results_page():
    st.sidebar.header("Your Role")
    st.sidebar.info(f"Currently logged in as: **{st.session_state.selected_role.capitalize()}**")
    if st.sidebar.button("Change Role"):
        st.session_state.page = 'role_selection'
        st.session_state.search_query_input = "" # Clear query when changing role
        st.session_state.current_search_results = [] # Clear results
        # Removed st.experimental_rerun()

    st.title(f"🔍 Search Documents (Role: {st.session_state.selected_role.capitalize()})")
    st.markdown("### Enter your query or select from common queries below.")

    # Option to select a common query
    common_queries = sample_queries_by_role.get(st.session_state.selected_role, [])
    if common_queries:
        selected_common_query = st.selectbox(
            "Choose a common query:",
            [""] + common_queries, # Add empty option
            index=0,
            key="common_query_select" # Unique key for the selectbox
        )
        if selected_common_query:
            st.session_state.search_query_input = selected_common_query # Update text input with selected common query

    # Text input for custom query
    query_text = st.text_input(
        "Or enter a custom query:",
        value=st.session_state.search_query_input,
        key="custom_query_input" # Unique key for the text input
    )
    st.session_state.search_query_input = query_text # Keep session state updated

    search_button_col, _ = st.columns([1, 4])
    with search_button_col:
        if st.button("Search Documents", use_container_width=True):
            if st.session_state.search_query_input:
                with st.spinner("Searching for relevant documents..."):
                    results = role_based_search(st.session_state.search_query_input, st.session_state.selected_role)
                    st.session_state.current_search_results = results # Store results in session state
                st.subheader(f"Results for your Search:")
            else:
                st.error("Please enter a query to search.")

    # Display results if any are stored in session state
    if st.session_state.current_search_results:
        for i, res in enumerate(st.session_state.current_search_results):
            with st.expander(f"📄 Document {i+1} from {res['source']}"):
                st.markdown(
                    f"**Source File:** `{res['source']}`\n\n"
                    f"**Accessible by:** {', '.join([r.capitalize() for r in res['roles']])}\n\n"
                    f"**Content:**\n{res['document']}"
                )
            st.markdown("---")
    elif st.session_state.search_query_input and st.button("Search Documents"): # Only show info if a search was performed and yielded no results
         st.info("No relevant documents found for your query and role. Try a different query or role.")


# --- Main App Logic ---
if st.session_state.page == 'role_selection':
    show_role_selection_page()
elif st.session_state.page == 'query_and_results':
    show_query_and_results_page()

In [ ]:
ngrok.kill()

# Start ngrok tunnel for Streamlit port 8501
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

In [ ]:
import requests
import time

# Function to check if Streamlit is running
def check_streamlit_status(port=8501, retries=5, delay=5):
    print(f"Checking Streamlit server on port {port}...")
    for i in range(retries):
        try:
            response = requests.get(f"http://localhost:{port}", timeout=1)
            if response.status_code == 200:
                print(f"Streamlit server is running on port {port}!")
                return True
        except requests.exceptions.ConnectionError:
            print(f"Attempt {i + 1}/{retries}: Streamlit server not yet ready. Retrying in {delay} seconds...")
            time.sleep(delay)
        except requests.exceptions.Timeout:
            print(f"Attempt {i + 1}/{retries}: Connection timed out. Retrying in {delay} seconds...")
            time.sleep(delay)
    print("Streamlit server did not start within the expected time.")
    return False

# Stop any existing Streamlit process before starting a new one
!kill $(lsof -t -i:8501) 2>/dev/null || true

# Start Streamlit in the background
get_ipython().run_line_magic('shell', 'streamlit run app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false &')

# Verify if Streamlit has started
if check_streamlit_status():
    print("Streamlit app is confirmed to be running. You can now proceed to run the ngrok cell.")
else:
    print("Failed to start Streamlit app. Please check the logs for errors.")